# Qualidade do agrupamento (ARI) vs. resultado da classificação

Pergunta: **um agrupamento melhor leva a uma classificação melhor?**

- **ARI** entre os clusters verdadeiros do `make_classification` e a partição rígida (`argmax` da pertinência) que o modelo encontrou nos pontos de treino de cada fold.
- **Acurácia** (e AUC) da classificação final.
- Eixo X: valor do parâmetro (**c** com `c_override`, ou **m** com `fuzzy_m`).

Entrada: o `results.csv` de um experimento rodado com `Suggested_c` e `c2`...`c10` (e, opcionalmente, `Fuzzy_m*`), com várias repetições Monte Carlo (`dataset_idx`). Cada ponto dos gráficos é a média entre os datasets, e a barra é o desvio entre eles.

In [ ]:
import re
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import spearmanr

## Dados

Preencha `EXPERIMENT` com o nome do experimento (pasta em `outputs/`). Com `RUN = None` usa a execução mais recente; para fixar uma, passe o nome da pasta de timestamp.

In [ ]:
OUTPUTS = Path('../outputs')
EXPERIMENT = 'control'
RUN = None  # ex.: '20261028_153000'; None = mais recente

run_dir = OUTPUTS / EXPERIMENT / (RUN or sorted(p.name for p in (OUTPUTS / EXPERIMENT).iterdir())[-1])
print('Lendo', run_dir)

df = pd.read_csv(run_dir / 'results.csv')
if 'dataset_idx' not in df:  # CSV de antes das repetições Monte Carlo
    df['dataset_idx'] = 0
print(f"{df['dataset_idx'].nunique()} dataset(s), modelos: {list(df['model'].unique())}")

In [ ]:
def parse_model(name):
    """(familia, valor do parametro) a partir do nome do modelo no registry."""
    if name == 'Suggested_c':
        return 'suggested', np.nan
    if name.startswith('c') and name[1:].isdigit():
        return 'c', float(name[1:])
    if name.startswith('Fuzzy_m'):
        return 'm', float(name[len('Fuzzy_m'):])
    return 'other', np.nan


df[['family', 'value']] = df['model'].apply(lambda n: pd.Series(parse_model(n)))
df = df[df['family'] != 'other'].copy()
df.groupby('family')['model'].unique()

## Curva ARI e acurácia contra o parâmetro

O ARI usa o eixo da esquerda e a acurácia o da direita. A faixa tracejada é o `Suggested_c`, o modelo com a escolha automática de c, que serve de referência (o c dele varia por dataset, então não tem um ponto no eixo X).

In [ ]:
ARI_COLOR, ACC_COLOR = '#2a6fdb', '#d9622b'


def mean_std(frame, col):
    g = frame.groupby('value')[col]
    return g.mean(), g.std().fillna(0)


def plot_param_curve(df, family, xlabel, acc_col='accuracy_mean'):
    sub = df[df['family'] == family]
    if sub.empty:
        print(f'Sem modelos da familia {family!r} neste CSV.')
        return
    ref = df[df['family'] == 'suggested']

    fig, ax_ari = plt.subplots(figsize=(8, 4.6))
    ax_acc = ax_ari.twinx()

    for ax, col, color, marker, label in [
        (ax_ari, 'ari_mean', ARI_COLOR, 'o', 'ARI do agrupamento'),
        (ax_acc, acc_col, ACC_COLOR, 's', 'Acuracia'),
    ]:
        m, s = mean_std(sub, col)
        ax.errorbar(m.index, m.values, yerr=s.values, color=color, marker=marker,
                    capsize=3, lw=2, label=label)
        if not ref.empty:
            rm, rs = ref[col].mean(), ref[col].std()
            ax.axhline(rm, color=color, ls='--', lw=1)
            ax.axhspan(rm - rs, rm + rs, color=color, alpha=0.08)
        ax.set_ylabel(label, color=color)
        ax.tick_params(axis='y', labelcolor=color)

    ax_ari.set_xlabel(xlabel)
    ax_ari.set_xticks(sorted(sub['value'].unique()))
    ax_ari.set_title(f'ARI e acuracia vs {xlabel}  (tracejado = Suggested_c)')
    ax_ari.grid(alpha=0.25)
    fig.tight_layout()
    return fig


plot_param_curve(df, 'c', 'c (numero de clusters)');

In [ ]:
plot_param_curve(df, 'm', 'm (fuzzificador)');

## Correspondência: o ARI acompanha a acurácia?

Cada ponto é um par (dataset, valor do parâmetro). A correlação **agregada** mistura datasets fáceis e difíceis, então ela pode refletir a dificuldade do dataset e não o efeito do parâmetro. Por isso também calculo a correlação **dentro de cada dataset** (só o parâmetro varia) e tiro a média.

In [ ]:
def correspondence(df, family, xlabel, acc_col='accuracy_mean'):
    sub = df[df['family'] == family].dropna(subset=['ari_mean', acc_col])
    if sub.empty:
        print(f'Sem modelos da familia {family!r} neste CSV.')
        return

    pooled = spearmanr(sub['ari_mean'], sub[acc_col])
    per_ds = []
    for _, g in sub.groupby('dataset_idx'):
        if g['value'].nunique() > 2:
            per_ds.append(spearmanr(g['ari_mean'], g[acc_col])[0])
    within = float(np.nanmean(per_ds)) if per_ds else np.nan

    fig, ax = plt.subplots(figsize=(5.6, 4.6))
    sc = ax.scatter(sub['ari_mean'], sub[acc_col], c=sub['value'], cmap='viridis', s=36, alpha=0.85)
    fig.colorbar(sc, ax=ax, label=xlabel)
    ax.set_xlabel('ARI do agrupamento')
    ax.set_ylabel('Acuracia')
    ax.set_title(f'Spearman agregado = {pooled[0]:.2f} (p = {pooled[1]:.3f})\n'
                 f'media dentro de cada dataset = {within:.2f}')
    ax.grid(alpha=0.25)
    fig.tight_layout()
    return fig


correspondence(df, 'c', 'c');

In [ ]:
correspondence(df, 'm', 'm');

## Tabela de apoio

In [ ]:
def summary_table(df, family):
    sub = df[df['family'] == family]
    if sub.empty:
        return None
    cols = ['ari_mean', 'accuracy_mean', 'auc_mean', 'c_mean']
    g = sub.groupby('value')[cols].agg(['mean', 'std']).round(3)
    g.columns = [f'{a}_{b}' for a, b in g.columns]
    return g


summary_table(df, 'c')

In [ ]:
# c escolhido pelo modelo automatico em cada dataset (media dos folds)
df[df['family'] == 'suggested'][['dataset_idx', 'c_mean', 'ari_mean', 'accuracy_mean']].round(3)

## Notas

(Preencher depois de rodar: o pico do ARI coincide com o pico de acurácia? O c sugerido cai perto do melhor c para a acurácia? Para qual c o ARI é maior e o que acontece com a classificação?)